# CRAG Reproduction & Analysis — Complete Pipeline
**Corrective Retrieval-Augmented Generation** reproduced on PopQA with a local quantized Qwen2.5-7B backend.

This notebook runs the full study end to end:
1. Setup + local model + embeddings
2. Data (PopQA) + dev/eval splits
3. CRAG components (calibrated grader, rewriter, web search, generator, re-ranking)
4. The CRAG state machine (LangGraph)
5. Two corpora: **easy** (eval subjects only) and **hard** (eval + 300 distractors)
6. Experiments: Vanilla RAG vs CRAG on both corpora
7. Comparison analysis


## Section 0 — Installs


In [1]:
# Install everything (run once, then RESTART runtime)
!pip -q install "langchain>=0.3,<0.4" "langchain-community>=0.3,<0.4" "langgraph>=0.2" \
  "langchain-huggingface" "faiss-cpu" "sentence-transformers" "datasets" "rank_bm25" "gradio"
!pip -q install bitsandbytes accelerate
!pip -q install ddgs hf_transfer
print("Installs done. Now: Runtime > Restart session, then run from Section 1.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.

## Section 1 — Login, warnings, imports, config

In [2]:
import os, warnings, logging
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

# Optional but recommended: add HF_TOKEN to Colab secrets for faster downloads
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("HF login OK")
except Exception as e:
    print("HF token not set (downloads still work, just slower)")

HF login OK


In [3]:
import json, re, time, random
import numpy as np
import torch, requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline, ChatHuggingFace
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from ddgs import DDGS

EMBED_MODEL   = "BAAI/bge-small-en-v1.5"
CHUNK_SIZE    = 256      # tokens (chunk-size sweep variable)
CHUNK_OVERLAP = 32
TOP_K         = 5
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


## Section 2 — Local LLM backend (quantized Qwen2.5-7B on GPU)
First run downloads ~5.5 GB.

In [4]:
MODEL_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"   # pre-quantized 4-bit
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map={"": 0})  # force GPU 0

gen_pipe = pipeline("text-generation", model=model, tokenizer=tok,
                    max_new_tokens=32, do_sample=False,        # greedy = deterministic
                    return_full_text=False, repetition_penalty=1.1)
llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=gen_pipe))

print("device:", next(model.parameters()).device,
      "| VRAM:", round(torch.cuda.memory_allocated()/1e9, 1), "GB")
print(llm.invoke("Reply with exactly: CRAG backend OK").content)

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'repetition_penalty', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


device: cuda:0 | VRAM: 5.5 GB


[transformers] Both `max_new_tokens` (=32) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


CRAG backend OK


In [5]:
# Embedding model for dense retrieval + re-ranking
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL,
                                   encode_kwargs={"normalize_embeddings": True})
print("Embeddings ready.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings ready.


## Section 3 — Data: PopQA, scorer, dev/eval splits

In [6]:
ds = load_dataset("akariasai/PopQA")
data = ds["test"]
print("PopQA loaded:", len(data), "questions")

def parse_gold_answers(ex):
    # possible_answers is a JSON-encoded STRING -> list
    return [a.strip() for a in json.loads(ex["possible_answers"]) if a and a.strip()]

def normalize(t): return t.lower().strip()

def exact_match(pred, gold):
    # PopQA accuracy: any gold answer is a substring of the prediction
    p = normalize(pred)
    return any(normalize(g) in p for g in gold)

random.seed(42)
dev_set = [data[i] for i in random.sample(range(len(data)), 50)]   # for tuning

random.seed(123)
dev_ids = {e["id"] for e in dev_set}
pool = [data[i] for i in range(len(data)) if data[i]["id"] not in dev_ids]
eval_set = random.sample(pool, 100)                                # held-out
print("dev:", len(dev_set), "| eval:", len(eval_set), "(disjoint)")

README.md:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


test.tsv:   0%|          | 0.00/5.21M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/14267 [00:00<?, ? examples/s]

PopQA loaded: 14267 questions
dev: 50 | eval: 100 (disjoint)


## Section 4 — Reusable corpus builders (fetch Wikipedia + chunk)

In [7]:
session = requests.Session()
session.headers.update({"User-Agent": "CRAG-course-project/1.0 (educational)"})
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=1.5, status_forcelist=[429,500,502,503,504],
    respect_retry_after_header=True)))

def fetch_wikipedia_extract(title):
    r = session.get("https://en.wikipedia.org/w/api.php",
        params={"action":"query","prop":"extracts","explaintext":1,
                "titles":title,"format":"json","redirects":1}, timeout=30)
    r.raise_for_status()
    return next(iter(r.json()["query"]["pages"].values())).get("extract", "")

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    AutoTokenizer.from_pretrained(EMBED_MODEL),
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

def fetch_articles(titles, pause=0.5):
    arts = {}
    titles = sorted(set(titles))
    print(f"Fetching {len(titles)} articles...")
    for i, t in enumerate(titles, 1):
        try:
            txt = fetch_wikipedia_extract(t)
            if txt: arts[t] = txt
        except Exception:
            pass
        time.sleep(pause)
        if i % 50 == 0: print(f"  ...{i} done")
    print(f"Fetched {len(arts)}/{len(titles)}.")
    return arts

def chunk_articles(arts):
    docs = []
    for title, text in arts.items():
        for ch in splitter.split_text(text):
            docs.append(Document(page_content=ch,
                        metadata={"title": title, "source": "wikipedia"}))
    return docs

## Section 5 — CRAG components
Calibrated grader (loosened to avoid over-rejection), query rewriter, web search, generator, and the re-ranking helper.

In [8]:
# --- Retrieval evaluator (CALIBRATED: prefers Correct/Ambiguous when unsure) ---
GRADER_PROMPT = """You are a retrieval evaluator. Decide whether the retrieved documents
are likely to help answer the question about the entity named in it.

Question:
{question}

Retrieved documents:
{documents}

Guidance:
- Score "Correct" if the documents are about the right entity AND plausibly contain the answer.
- Score "Ambiguous" only if the documents are about the right entity but clearly lack the specific fact.
- Score "Incorrect" ONLY if the documents are about a clearly DIFFERENT entity/topic, or contain
  nothing related to the question. When unsure, prefer "Correct" or "Ambiguous" over "Incorrect".

Respond with ONLY a JSON object and nothing else:
{{"reasoning": "<one short sentence>", "score": "<Correct|Ambiguous|Incorrect>"}}"""

_VALID = {"Correct", "Ambiguous", "Incorrect"}

def _extract_json(t):
    m = re.search(r"\{.*\}", t, re.DOTALL)
    if not m: return None
    try: return json.loads(m.group(0))
    except Exception: return None

def grade_retrieval(question, docs):
    dt = "\n\n".join(f"[{i+1}] {d.page_content}" for i, d in enumerate(docs))
    resp = llm.invoke(GRADER_PROMPT.format(question=question, documents=dt))
    p = _extract_json(resp.content)
    if p and p.get("score") in _VALID:
        return {"score": p["score"], "reasoning": p.get("reasoning", "")}
    low = resp.content.lower()
    for lab in ["incorrect", "ambiguous", "correct"]:
        if lab in low: return {"score": lab.capitalize(), "reasoning": "fallback"}
    return {"score": "Ambiguous", "reasoning": "default"}

In [9]:
# --- Query rewriter + web search (corrective source) ---
REWRITE_PROMPT = """You are rewriting a question into a web search query that will
find the answer. Keep ALL named entities exactly as written, and add a disambiguating
word for the entity type if implied (e.g. song, film, book, person, city).
Return ONLY the search query, nothing else.

Question: {question}
Search query:"""

def rewrite_query(question):
    return llm.invoke(REWRITE_PROMPT.format(question=question)).content.strip().strip('"')

def web_search(query, max_results=5):
    docs = []
    try:
        with DDGS() as d:
            for r in d.text(query, max_results=max_results):
                docs.append(Document(
                    page_content=f"{r.get('title','')}\n{r.get('body','')}",
                    metadata={"title": r.get("title",""), "source": "web",
                              "url": r.get("href","")}))
    except Exception as e:
        print("web_search error:", e)
    return docs

In [10]:
# --- Generator + re-ranking ---
GENERATOR_PROMPT = """Answer the question using ONLY the context below.
Be concise — give just the answer (a name, term, or short phrase), not a full sentence.
If the context does not contain the answer, reply exactly: I don't know.

Context:
{context}

Question: {question}
Answer:"""

def generate_answer(question, docs):
    ctx = "\n\n".join(f"[{i+1}] {d.page_content}" for i, d in enumerate(docs))
    return llm.invoke(GENERATOR_PROMPT.format(context=ctx, question=question)).content.strip()

def rerank_docs(question, docs, top_k=5):
    """Re-rank docs by embedding similarity to the question; keep top_k (cuts noise)."""
    if not docs: return docs
    qv = np.array(embeddings.embed_query(question))
    dv = np.array(embeddings.embed_documents([d.page_content for d in docs]))
    order = np.argsort(dv @ qv)[::-1][:top_k]
    return [docs[i] for i in order]

## Section 6 — CRAG state machine (LangGraph)
`retrieve -> grade -> {Correct: generate | else: websearch(augment + re-rank) -> generate}`.
`ACTIVE_RETRIEVER` is swappable so the same graph runs over either corpus.

In [11]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

class CRAGState(TypedDict):
    question: str
    documents: List[Document]
    grade: str
    generation: str
    steps: List[str]

ACTIVE_RETRIEVER = None   # set to eval_dense or hard_dense before running

def node_retrieve(s):
    return {"documents": ACTIVE_RETRIEVER.invoke(s["question"]),
            "steps": s.get("steps", []) + ["retrieve"]}

def node_grade(s):
    r = grade_retrieval(s["question"], s["documents"])
    return {"grade": r["score"], "steps": s["steps"] + [f"grade={r['score']}"]}

def node_websearch(s):
    web = web_search(rewrite_query(s["question"]))
    combined = s["documents"] + web                       # augment (keep local)
    docs = rerank_docs(s["question"], combined, top_k=TOP_K)   # trim noise
    lab = "websearch(augment)" if s["grade"] == "Ambiguous" else "websearch(incorrect+augment)"
    return {"documents": docs, "steps": s["steps"] + [lab + "+rerank"]}

def node_generate(s):
    return {"generation": generate_answer(s["question"], s["documents"]),
            "steps": s["steps"] + ["generate"]}

def route_after_grade(s):
    return "generate" if s["grade"] == "Correct" else "websearch"

def build_crag_app():
    b = StateGraph(CRAGState)
    b.add_node("retrieve", node_retrieve)
    b.add_node("grade", node_grade)
    b.add_node("websearch", node_websearch)
    b.add_node("generate", node_generate)
    b.add_edge(START, "retrieve")
    b.add_edge("retrieve", "grade")
    b.add_conditional_edges("grade", route_after_grade,
                            {"generate": "generate", "websearch": "websearch"})
    b.add_edge("websearch", "generate")
    b.add_edge("generate", END)
    return b.compile()

print("Graph functions defined.")

Graph functions defined.


/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## Section 7 — Build the two corpora
**Easy** = 100 eval subjects. **Hard** = eval subjects + 300 distractor articles (~2-3 min fetch + embed).

In [12]:
# Easy corpus: just the eval subjects
eval_articles = fetch_articles([e["s_wiki_title"] for e in eval_set])
eval_docs = chunk_articles(eval_articles)
eval_vs = FAISS.from_documents(eval_docs, embeddings)
eval_dense = eval_vs.as_retriever(search_kwargs={"k": TOP_K})
print("EASY corpus:", len(eval_docs), "chunks from", len(eval_articles), "articles")

Fetching 100 articles...
  ...50 done


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (659 > 512). Running this sequence through the model will result in indexing errors


  ...100 done
Fetched 100/100.
EASY corpus: 1747 chunks from 100 articles


## Section 8 — Chunk Size Sweep


In [13]:
# Subset eval harness (needed by the chunk sweep)
def run_eval_subset(retriever, label, path, questions):
    global ACTIVE_RETRIEVER, crag_app
    ACTIVE_RETRIEVER = retriever
    crag_app = build_crag_app()
    def vanilla(q): return generate_answer(q, retriever.invoke(q))
    res, t0 = {}, time.time()
    for i, ex in enumerate(questions, 1):
        q, gold = ex["question"], parse_gold_answers(ex)
        ba = vanilla(q)
        co = crag_app.invoke({"question": q, "steps": []})
        res[str(ex["id"])] = {
            "question": q, "gold": gold,
            "baseline_ans": ba, "baseline_correct": exact_match(ba, gold),
            "crag_ans": co["generation"], "crag_correct": exact_match(co["generation"], gold),
            "path": co["steps"],
        }
        if i % 10 == 0:
            with open(path, "w") as f: json.dump(res, f)
            print(f"  [{label}] {i}/{len(questions)} | {time.time()-t0:.0f}s")
    with open(path, "w") as f: json.dump(res, f)
    print(f"[{label}] DONE")
    return res

In [14]:
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

In [15]:
# CHUNK-SIZE SWEEP — run in the second account after Sections 0-7
# Reuses eval_articles + eval_set; builds one index at a time to spare RAM.
import gc

def chunk_at_size(articles, size, overlap):
    sp = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        AutoTokenizer.from_pretrained(EMBED_MODEL),
        chunk_size=size, chunk_overlap=overlap)
    docs = []
    for title, text in articles.items():
        for ch in sp.split_text(text):
            docs.append(Document(page_content=ch,
                        metadata={"title": title, "source": "wikipedia"}))
    return docs

chunk_sweep = {}
for size in [128, 256, 512]:
    print(f"\n===== chunk_size = {size} =====")
    docs = chunk_at_size(eval_articles, size, overlap=max(16, size//8))
    vs = FAISS.from_documents(docs, embeddings)
    retr = vs.as_retriever(search_kwargs={"k": TOP_K})
    res = run_eval_subset(retr, f"chunk={size}", f"/content/results_chunk_{size}.json",
                          eval_set[:50])
    n = len(res)
    chunk_sweep[size] = (sum(r["baseline_correct"] for r in res.values())/n,
                         sum(r["crag_correct"] for r in res.values())/n)
    # free this index before the next size — prevents RAM creep
    del docs, vs, retr; gc.collect()

print(f"\n{'chunk':>6}{'Vanilla':>9}{'CRAG':>8}")
for size, (v, c) in chunk_sweep.items():
    print(f"{size:>6}{v:>8.0%}{c:>8.0%}")


===== chunk_size = 128 =====
  [chunk=128] 10/50 | 157s
  [chunk=128] 20/50 | 322s
  [chunk=128] 30/50 | 479s
  [chunk=128] 40/50 | 633s
  [chunk=128] 50/50 | 790s
[chunk=128] DONE

===== chunk_size = 256 =====
  [chunk=256] 10/50 | 229s
  [chunk=256] 20/50 | 481s
  [chunk=256] 30/50 | 729s
  [chunk=256] 40/50 | 969s
  [chunk=256] 50/50 | 1212s
[chunk=256] DONE

===== chunk_size = 512 =====
  [chunk=512] 10/50 | 394s
  [chunk=512] 20/50 | 833s
  [chunk=512] 30/50 | 1230s
  [chunk=512] 40/50 | 1676s
  [chunk=512] 50/50 | 2098s
[chunk=512] DONE

 chunk  Vanilla    CRAG
   128     60%     62%
   256     60%     62%
   512     70%     64%
